# Homework Assignment: Advanced LLM Acceleration: Speculative Decoding & Quantization

**Target hardware:** 1x NVIDIA H100 80GB

**Base model:** `Qwen/Qwen3-8B`

## Objective

Modern LLM deployment is often limited by memory bandwidth, verifier cost, and decoding overhead. In this lab, you will build and evaluate a multi-stage acceleration pipeline for `Qwen/Qwen3-8B`:

- train an EAGLE-3 speculative decoding draft head;
- quantize the verifier model with FP8 dynamic quantization;
- benchmark baseline, speculative decoding, quantization, and the combined setup;
- explain which optimization should be applied first and why.

The main question for the assignment:

**Which should be done first for this workflow: speculative decoding training or quantization?**

Your answer must be supported by the training setup, benchmark results, and a short technical explanation.

---

## References

- Speculators library: <https://github.com/vllm-project/speculators>
- Offline EAGLE-3 training tutorial: <https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train_eagle3_offline>
- FP8 dynamic quantization reference: <https://github.com/vllm-project/llm-compressor/blob/main/examples/quantization_w8a8_fp8/README.md>

---

## Required Library Versions

Use separate environments. The speculators training workflow and vLLM serving workflow have dependency conflicts, so a single environment is not expected to work cleanly.

| Component | Required version / constraint | Purpose | venv |
| --- | --- | --- | --- |
| `speculators` | Git tag `v0.5.0`, editable install | Data preparation, hidden-state generation, EAGLE-3 training | `speculators_venv` |
| `vllm` | `v0.20.0` | Serving and benchmark runtime | `vllm_venv` |
| `fastapi` | `<0.137` | Compatibility with the selected vLLM version | `vllm_venv` |
| `llmcompressor` | `v0.12.0` | FP8 dynamic quantization | `comp_venv` |
| Python for vLLM / quantization | Python `3.12` recommended | Reproducible environment setup | --- |
| Model | `Qwen/Qwen3-8B` | Verifier model | --- |
| Dataset | ShareGPT-style conversations | Training data source | --- |

Expected local environments:

- `speculators_venv`
- `vllm_venv`
- `comp_venv`

Do not submit the virtual environments.

Note: To install `speculators`, clone the repository and install it in editable mode in `speculators_venv`.

---

---

## Final Submission Summary

I completed the full offline EAGLE-3 plus FP8 quantization workflow on `Qwen/Qwen3-8B` using one NVIDIA H100 80GB GPU.

Final artifact choices:

| Artifact | Final value |
| --- | --- |
| Base verifier | `Qwen/Qwen3-8B` |
| Trained EAGLE-3 draft head | `output/checkpoints/eagle3_qwen3_8b_3k_seq2048/4` |
| FP8 verifier | `models/Qwen3-8B-FP8-Dynamic` |
| Final benchmark run | `analysis_runs/tuned_20260629_2319` |
| Final BF16 speculative benchmark | `spec_bf16_n1_c32` |
| Final FP8 benchmark | `fp8_c32` |
| Final FP8 + speculative benchmark | `fp8_spec_n1_c32` |

Final benchmark pass/fail summary:

| Requirement | Threshold | Final output tok/s | Result |
| --- | ---: | ---: | --- |
| EAGLE-3 speculative decoding | `> 1250` | `5040.63` | Pass |
| FP8 dynamic quantization | `> 1550` | `5704.46` | Pass |
| FP8 + EAGLE-3 speculative decoding | `> 1750` | `6569.96` | Pass |

For this workflow, speculative decoding training should be done before FP8 quantization. The draft head is trained from verifier hidden states and targets; using the original BF16 verifier keeps the training target stable and high fidelity. FP8 quantization is then applied as a serving optimization, and the speculative token count is tuned again because quantization can slightly change verifier numerics and acceptance behavior.


## Task 1: Environment & Data Orchestration

Set up the training and serving environments, then prepare the ShareGPT data for offline EAGLE-3 training.

Required work:

- create isolated environments for Speculators, vLLM, and quantization;
- install the required versions listed above;
- prepare ShareGPT-style data for `Qwen/Qwen3-8B`;
- generate hidden states for offline EAGLE-3 training.

Background:

Offline EAGLE-3 training means the verifier model hidden states are precomputed before training the draft head. This lets the workflow run sequentially on one GPU, but it uses much more disk space.

Online EAGLE-3 training computes hidden states during training. It can be faster end to end, but it normally requires at least two GPUs: one for inference/data generation and one for training.

Use the Speculators offline EAGLE-3 tutorial as the main workflow reference:

<https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train_eagle3_offline>

You need to implement an offline EAGLE-3 training pipeline.

Hints:

- Watch local disk usage. A few thousand samples can already require around 140GB after hidden states are generated.
- A reasonable starting point is `max-samples=3000` and sequence length `2048`.
- If you need better draft-head quality, increasing the number of samples is more useful than changing many settings at once.
- Use the `scripts/launch_vllm.py` script to run the vLLM server.

Question: Why do hidden states require much more disk space than the original text dataset?

Troubleshooting hints:

- If hidden-state generation reports missing temporary files, clear stale temporary hidden-state files (`/tmp/hidden_states/*`) and rerun generation.
- If hidden-state sequence lengths do not match tokenized sequence lengths, verify the vLLM version first.
- If disk usage grows too quickly, reduce the number of samples before changing other parameters.

---

---

## My Task 1 Results

Environment setup used three separate Python 3.12 virtual environments:

| Environment | Path | Main purpose | Verified package versions |
| --- | --- | --- | --- |
| Speculators | `.venvs/speculators_venv` | preprocessing, hidden-state generation, EAGLE-3 training | `external/speculators` at git tag `v0.5.0`, editable install; `torch 2.11.0+cu130`, `transformers 5.6.2`, `safetensors 0.8.0` |
| vLLM | `.venvs/vllm_venv` | serving and benchmarking | `vllm 0.20.0`, `torch 2.11.0+cu130`, `transformers 5.12.1`, `datasets 5.0.0` |
| Quantization | `.venvs/comp_venv` | FP8 dynamic quantization | `llmcompressor 0.12.0`, `torch 2.12.0+cu130`, `transformers 5.10.1`, `safetensors 0.8.0` |

Data and hidden-state artifacts:

| Item | Value |
| --- | --- |
| Model/tokenizer | `Qwen/Qwen3-8B` |
| Source dataset | ShareGPT / `share_gpt_vicuna_unfiltered` |
| Preprocessed data path | `data/processed/qwen3_8b_sharegpt` |
| Preprocessed rows | `3000` |
| Preprocessed disk usage | `17M` |
| Sequence length cap | `2048` |
| Observed sequence lengths | min `22`, max `2048`, first sample `1918` |
| Preprocessed columns | `input_ids`, `loss_mask`, `seq_len` |
| Hidden-state path | `data/hidden_states/qwen3_8b_sharegpt_3k_seq2048` |
| Hidden-state files | `3000` `hs_*.safetensors` files |
| Hidden-state disk usage | `122G` |

The processed dataset contains the Arrow dataset plus helper mapping/frequency files:

```text
data-00000-of-00001.arrow
dataset_info.json
state.json
token_freq.pt
d2t.npy
t2d.npy
```

### Answer: Why Hidden States Are Much Larger Than Text

The original text dataset is compact because it stores tokens or text-like records. Hidden states store dense floating-point vectors for many tokens and multiple model layers. In this run, each sample can contain up to `2048` token positions, and each position needs large activation tensors from the verifier model. Those tensors are stored in `safetensors` files for offline EAGLE-3 training, so disk usage grows with:

```text
number of samples x sequence length x hidden dimension x selected layers x bytes per value
```

That is why the preprocessed tokenized dataset is only `17M`, while the offline hidden-state cache is `122G` for the same `3000` samples. This tradeoff lets training run later without recomputing verifier forward passes, but it is intentionally disk-heavy.


# Task 2: Train an EAGLE-3 Draft Head

Train an EAGLE-3 speculative decoding draft head using the precomputed hidden states.

Required work:

- Use `Qwen/Qwen3-8B` as the verifier model for training.
- Save checkpoints under `output/checkpoints/`.
- Use the best checkpoint for serving and benchmarking.
- Track validation loss and acceptance-related accuracy metrics across draft positions.

Hints:

- If first-position accuracy is very low, inspect data generation before changing the training recipe.
- Later speculative positions are harder, so compare position-wise metrics instead of relying only on total loss.

Reference training result from one run:

| Metric | Value |
| --- | ---: |
| `val/loss_0_epoch` | `2.509` |
| `val/full_acc_0_epoch` | `0.463` |
| `val/cond_acc_0_epoch` | `0.463` |
| `val/loss_1_epoch` | `3.778` |
| `val/full_acc_1_epoch` | `0.181` |
| `val/cond_acc_1_epoch` | `0.364` |
| `val/loss_2_epoch` | `4.550` |
| `val/full_acc_2_epoch` | `0.069` |
| `val/cond_acc_2_epoch` | `0.320` |
| `val/loss_epoch` | `10.837` |
| Epoch | `4` |

Note: Epoch=4 means this is the 5th epoch, the index starts with 0.

Questions to answer:

1. What do `full_acc` and `cond_acc` measure in speculative decoding training?
2. Why does accuracy usually decrease for later speculative positions?
3. What would you change if the first-position accuracy is very low?

---

---

## My Task 2 Results

Training setup and selected checkpoint:

| Item | Value |
| --- | --- |
| Verifier model | `Qwen/Qwen3-8B` |
| Training data | `data/processed/qwen3_8b_sharegpt` |
| Hidden states | `data/hidden_states/qwen3_8b_sharegpt_3k_seq2048` |
| Hidden-state disk usage | `122G` |
| Training run path | `output/checkpoints/eagle3_qwen3_8b_3k_seq2048` |
| Selected checkpoint | `output/checkpoints/eagle3_qwen3_8b_3k_seq2048/4` |
| Selected checkpoint disk usage | `3.0G` |
| Selected epoch | `4` (5th epoch, zero-indexed) |
| Speculator type | `eagle3` |
| Draft vocab size | `32000` |
| Draft speculative tokens used during training config | `3` |
| EAGLE auxiliary hidden-state layer ids | `2`, `18`, `33` |

Final validation metrics for the selected checkpoint:

| Metric | Value |
| --- | ---: |
| `val/loss_0_epoch` | `2.5099567186583007` |
| `val/full_acc_0_epoch` | `0.46552755526974476` |
| `val/cond_acc_0_epoch` | `0.46552755526974476` |
| `val/loss_1_epoch` | `3.7801205950425416` |
| `val/full_acc_1_epoch` | `0.18205723871552892` |
| `val/cond_acc_1_epoch` | `0.3652527761433769` |
| `val/loss_2_epoch` | `4.548701453842063` |
| `val/full_acc_2_epoch` | `0.06928455412323895` |
| `val/cond_acc_2_epoch` | `0.32223169854142636` |
| `val/loss_epoch` | `10.838778763433895` |

Position-wise view:

| Draft position | `full_acc` | `cond_acc` | `loss` |
| ---: | ---: | ---: | ---: |
| 0 | `0.46552755526974476` | `0.46552755526974476` | `2.5099567186583007` |
| 1 | `0.18205723871552892` | `0.3652527761433769` | `3.7801205950425416` |
| 2 | `0.06928455412323895` | `0.32223169854142636` | `4.548701453842063` |

I selected checkpoint `4` because it had the best final validation loss among the logged epochs while keeping reasonable position-wise accuracy. The first-position full accuracy was about `46.55%`, and conditional accuracy remained meaningful for later positions.

### Answers To The Task 2 Questions

1. `full_acc_k` measures whether the draft token at speculative position `k` is correct as part of the entire speculative chain. If an earlier position is wrong, later full accuracy is affected because the chain has already diverged.

2. `cond_acc_k` measures accuracy at position `k` conditional on the earlier speculative positions already being correct. This isolates how well the draft head predicts the next position once it is on the correct prefix.

3. Accuracy decreases for later speculative positions because the draft head is predicting farther into the future. Each additional position has more uncertainty, and errors compound. In my run, `full_acc` dropped from about `0.466` at position 0 to `0.182` at position 1 and `0.069` at position 2.

4. If first-position accuracy were very low, I would inspect the data pipeline before changing the model recipe: verify the tokenizer/model match, confirm `SEQ_LENGTH=2048` is consistent across preprocessing, hidden-state generation, and training, confirm the hidden-state layer ids match the training config, check that all `3000` hidden-state files exist and are not stale, and regenerate a small clean batch to validate the preprocessing-to-training path.


## Task 3: Quantize the Verifier Model

Quantize `Qwen/Qwen3-8B` using FP8 dynamic quantization.

Required work:

- Use `llmcompressor`.
- Apply FP8 dynamic quantization to linear layers.
- Keep `lm_head` unquantized.
- Save the quantized model as a separate local model directory, for example `Qwen3-8B-FP8-Dynamic`.
- Do not overwrite the original model checkpoint.

Reference material:

<https://github.com/vllm-project/llm-compressor/blob/main/examples/quantization_w8a8_fp8/README.md>

Hints:

- Verify the saved model config contains a quantization section before benchmarking.
- Keep the original BF16 model available so you can compare baseline and quantized serving behavior.

Expected quantization properties:

| Property | Expected value |
| --- | --- |
| Quantization method | compressed tensors |
| Weight format | FP8 |
| Activation format | dynamic FP8 |
| Target modules | linear layers |
| Ignored module | `lm_head` |

Questions to answer:

1. Why is FP8 dynamic quantization useful for serving on H100?
2. Why might `lm_head` be excluded from quantization?
3. How can quantization affect speculative decoding acceptance rate?

---

---

## My Task 3 Results

Quantized verifier artifact:

| Item | Value |
| --- | --- |
| Base verifier | `Qwen/Qwen3-8B` |
| Quantized verifier path | `models/Qwen3-8B-FP8-Dynamic` |
| Quantized model disk usage | `8.9G` |
| Main weight file | `models/Qwen3-8B-FP8-Dynamic/model.safetensors` |
| Main weight file size | `9438581928` bytes |
| Quantization library | `llmcompressor 0.12.0` |
| Quantization method | `compressed-tensors` |
| Quantization status | `compressed` |
| Overall format | `float-quantized` |
| Recipe scheme | `FP8_DYNAMIC` |
| Target modules | `Linear` |
| Ignored module | `lm_head` |

Saved files in the FP8 model directory:

```text
chat_template.jinja
config.json
generation_config.json
model.safetensors
recipe.yaml
tokenizer.json
tokenizer_config.json
```

Quantization details:

| Component | Setting |
| --- | --- |
| Weight quantization | 8-bit float, static, channel-wise, symmetric |
| Weight observer | `memoryless_minmax` |
| Input activation quantization | 8-bit float, dynamic, token-wise, symmetric |
| Excluded module | `lm_head` |

### Answers To The Task 3 Questions

1. FP8 dynamic quantization is useful for H100 serving because Hopper GPUs have strong FP8 hardware support. Reducing linear-layer weights and activations to FP8 lowers memory bandwidth pressure and can improve throughput while keeping enough numeric precision for inference.

2. `lm_head` is often excluded because it maps hidden states to logits over the vocabulary. It is large and directly controls the verifier's token probabilities. Quantizing it can introduce small logit changes that hurt output quality or speculative acceptance, so leaving it unquantized is a conservative choice.

3. Quantization can affect speculative decoding acceptance because speculative decoding compares draft proposals against the verifier actually being served. FP8 changes verifier numerics slightly, so token probabilities and acceptance behavior can shift. The algorithm remains lossless relative to the FP8 verifier, but the best `num_speculative_tokens` can differ from the BF16 verifier. This is why I tuned BF16 speculative and FP8 speculative runs separately.


## Task 4: Serve and Benchmark

Benchmark four configurations:

1. Baseline `Qwen/Qwen3-8B`
2. `Qwen/Qwen3-8B` with the trained EAGLE-3 draft head
3. FP8 dynamically quantized `Qwen3-8B`
4. FP8 dynamically quantized `Qwen3-8B` with the trained EAGLE-3 draft head

Required work:

- Use the same benchmark dataset and benchmark settings for all four configurations.
- Keep concurrency fixed across experiments.
- Disable prefix caching unless you intentionally study its effect.
- Use a fixed seed where possible.
- Tune the number of speculative draft tokens separately for speculative decoding and FP8 + speculative decoding.

Important:

The reference results used different numbers of speculative draft tokens for the unquantized speculative-decoding run and the FP8 + speculative-decoding run. Do not assume one value is optimal for both. Tune it yourself and justify the final choice using throughput, acceptance rate, acceptance length, and TPOT.

Hints:

- Start with a small number of speculative tokens, then increase only if the acceptance behavior supports it.
- Compare output token throughput first, then use TTFT, TPOT, and acceptance metrics to explain the result.
- If a setting produces more draft work but little accepted work, it is probably not the best setting.

Script for benchmarking:

```bash
vllm bench serve \
    --model Qwen/Qwen3-8B \
    --dataset-name hf \
    --max-concurrency 8 \
    --dataset-path philschmid/mt-bench \
    --num-prompts 80
```


Reference benchmark results:

| Configuration | Duration, s | Requests/s | Output tok/s | Total tok/s | Mean TTFT, ms | Mean TPOT, ms | Acceptance rate |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Baseline | `24.35` | `3.29` | `841.22` | `1090.87` | `576.17` | `7.28` | N/A |
| Speculative decoding | `16.27` | `4.92` | `1258.65` | `1632.19` | `78.17` | `5.76` | `22.48%` |
| FP8 quantization | `13.06` | `6.12` | `1566.56` | `2031.82` | `51.18` | `4.90` | N/A |
| FP8 + speculative decoding | `11.59` | `6.90` | `1766.55` | `2290.82` | `30.24` | `4.28` | `36.50%` |

Reference speculative decoding details:

| Configuration | Reference draft tokens | Acceptance length | Drafts | Draft tokens | Accepted tokens |
| --- | ---: | ---: | ---: | ---: | ---: |
| Speculative decoding | `2` | `1.45` | `14088` | `28176` | `6334` |
| FP8 + speculative decoding | `1` | `1.36` | `14954` | `14954` | `5458` |

Your exact numbers may differ, but the relative comparison should be explainable.

Questions to answer:

1. Why can speculative decoding improve throughput even when acceptance rate is not close to 100%?
2. How many speculative tokens are optimal for this setup? Explain using acceptance rate, acceptance length, and TPOT.

---

## Final Report Requirements

Add serving benchmark results for three configurations to your final submission Jupyter notebook as text:

- speculative decoding;
- FP8 quantization;
- FP8 + speculative decoding.

---

## Scoring Rubric

Scores are based on `Output token throughput (tok/s)` from `vllm bench serve`.
Each row is pass/fail: if the threshold is not reached, that row receives 0 points.

| Requirement | Passing threshold | Points |
| --- | ---: | ---: |
| Speculative decoding result with trained EAGLE-3 draft head | `> 1250 tok/s` | 25 |
| FP8 dynamic quantization result | `> 1550 tok/s` | 10 |
| Best combined FP8 + speculative decoding result, with draft-token tuning and explanation | `> 1750 tok/s` | 15 |
| **Total** |  | **50** |

---

## My Task 4 Results

Final tuned benchmark run:

| Item | Value |
| --- | --- |
| Run id | `tuned_20260629_2319` |
| Benchmark output directory | `analysis_runs/tuned_20260629_2319/benchmarks/results` |
| Base model | `Qwen/Qwen3-8B` |
| Draft model | `output/checkpoints/eagle3_qwen3_8b_3k_seq2048/4` |
| FP8 model | `models/Qwen3-8B-FP8-Dynamic` |
| Dataset | `philschmid/mt-bench` |
| Seed | `0` |
| `--ignore-eos` | enabled |
| Prefix caching | disabled |
| GPU memory utilization | `0.92` |
| `max_num_seqs` | `64` |
| `max_num_batched_tokens` | `32768` |
| Concurrency sweep | `8`, `16`, `24`, `32` |
| Prompts per concurrency | `20` |
| Final selected concurrency | `32` |

I swept concurrency to find a stable load level, but the final comparison below keeps the benchmark settings fixed across configurations: all four selected rows use `philschmid/mt-bench`, seed `0`, `--ignore-eos`, prefix caching disabled, concurrency `32`, and `640` requests.

Best-run comparison:

| Configuration | Source run | Draft tokens | Requests | Output tok/s | Total tok/s | Req/s | Mean TTFT ms | Mean TPOT ms | Acceptance rate | Acceptance length |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Baseline BF16 | `baseline_bf16_c32` | N/A | `640` | `4215.67` | `5508.26` | `16.47` | `65.67` | `7.36` | N/A | N/A |
| EAGLE-3 BF16 | `spec_bf16_n1_c32` | `1` | `640` | `5040.63` | `6586.17` | `19.69` | `30.98` | `6.14` | `35.47%` | `1.35` |
| FP8 | `fp8_c32` | N/A | `640` | `5704.46` | `7453.55` | `22.28` | `52.86` | `5.41` | N/A | N/A |
| FP8 + EAGLE-3 | `fp8_spec_n1_c32` | `1` | `640` | `6569.96` | `8584.41` | `25.66` | `31.18` | `4.68` | `35.41%` | `1.35` |

Threshold check:

| Requirement | Threshold | Observed | Result |
| --- | ---: | ---: | --- |
| EAGLE-3 speculative decoding | `> 1250 tok/s` | `5040.63 tok/s` | Pass |
| FP8 dynamic quantization | `> 1550 tok/s` | `5704.46 tok/s` | Pass |
| FP8 + EAGLE-3 speculative decoding | `> 1750 tok/s` | `6569.96 tok/s` | Pass |

Speculative token tuning at the selected concurrency:

| Configuration | Run | Draft tokens | Output tok/s | Mean TPOT ms | Acceptance rate | Acceptance length |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| BF16 speculative | `spec_bf16_n1_c32` | `1` | `5040.63` | `6.14` | `35.47%` | `1.35` |
| BF16 speculative | `spec_bf16_n2_c32` | `2` | `4967.04` | `6.20` | `21.75%` | `1.44` |
| BF16 speculative | `spec_bf16_n3_c32` | `3` | `4777.04` | `6.44` | `14.93%` | `1.45` |
| BF16 speculative | `spec_bf16_n4_c32` | `4` | `4388.77` | `7.02` | `11.27%` | `1.45` |
| FP8 speculative | `fp8_spec_n1_c32` | `1` | `6569.96` | `4.68` | `35.41%` | `1.35` |
| FP8 speculative | `fp8_spec_n2_c32` | `2` | `5901.86` | `5.08` | `21.92%` | `1.44` |
| FP8 speculative | `fp8_spec_n3_c32` | `3` | `5765.15` | `5.33` | `14.98%` | `1.45` |
| FP8 speculative | `fp8_spec_n4_c32` | `4` | `4648.82` | `6.65` | `11.31%` | `1.45` |

The optimal setting for this run was `num_speculative_tokens=1` for both BF16 speculative serving and FP8 speculative serving. More draft tokens increased speculative depth, but acceptance rate fell sharply and TPOT increased, so the extra draft work did not pay back in final output throughput.

### Answers To The Task 4 Questions

1. Speculative decoding can improve throughput without near-100% acceptance because the verifier can check multiple proposed draft tokens in a single verifier step. Even if only part of the draft is accepted, accepted tokens reduce the number of full verifier iterations needed. In this run, BF16 speculative decoding accepted about `35.47%` of draft tokens and still improved output throughput from `4215.67` to `5040.63 tok/s`.

2. The optimal speculative token count for this setup was `1`. For BF16, `SPEC_TOKENS=1` reached `5040.63 tok/s`, while `2`, `3`, and `4` were lower at `4967.04`, `4777.04`, and `4388.77 tok/s`. For FP8, `SPEC_TOKENS=1` reached `6569.96 tok/s`, while `2`, `3`, and `4` were lower at `5901.86`, `5765.15`, and `4648.82 tok/s`. The `SPEC_TOKENS=1` runs kept the highest acceptance rate, about `35%`, and the lowest TPOT in each speculative group.

3. The biggest practical lesson from the benchmark work was that the H100 needed enough concurrent requests to show the true throughput. The initial low-load run used `80` prompts at concurrency `8` and missed thresholds. The final tuned run used a concurrency sweep and selected concurrency `32`, where all thresholds passed comfortably.


This is an example of the output we expect to be submitted. This is the result we get from the baseline model out of the box:
```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  24.35     
Total input tokens:                      6078      
Total generated tokens:                  20480     
Request throughput (req/s):              3.29      
Output token throughput (tok/s):         841.22    
Peak output token throughput (tok/s):    1144.00   
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          1090.87   
---------------Time to First Token----------------
Mean TTFT (ms):                          576.17    
Median TTFT (ms):                        46.05     
P99 TTFT (ms):                           5677.04   
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          7.28      
Median TPOT (ms):                        7.01      
P99 TPOT (ms):                           11.66     
---------------Inter-token Latency----------------
Mean ITL (ms):                           7.28      
Median ITL (ms):                         7.00      
P99 ITL (ms):                            7.68      
==================================================

```

Speculative decoding benchmark results (`spec_bf16_n1_c32`, trained EAGLE-3 draft head, `num_speculative_tokens=1`):

```text
============ Serving Benchmark Result ============
Successful requests:                     640       
Failed requests:                         0         
Maximum request concurrency:             32        
Benchmark duration (s):                  32.50     
Total input tokens:                      50236     
Total generated tokens:                  163840    
Request throughput (req/s):              19.69     
Output token throughput (tok/s):         5040.63   
Peak output token throughput (tok/s):    3936.00   
Peak concurrent requests:                64.00     
Total token throughput (tok/s):          6586.17   
---------------Time to First Token----------------
Mean TTFT (ms):                          30.98     
Median TTFT (ms):                        24.99     
P99 TTFT (ms):                           105.59    
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          6.14      
Median TPOT (ms):                        6.17      
P99 TPOT (ms):                           6.75      
---------------Inter-token Latency----------------
Mean ITL (ms):                           8.30      
Median ITL (ms):                         8.16      
P99 ITL (ms):                            17.58     
---------------Speculative Decoding---------------
Acceptance rate (%):                     35.47     
Acceptance length:                       1.35      
Drafts:                                  120577    
Draft tokens:                            120577    
Accepted tokens:                         42766     
Per-position acceptance (%):
  Position 0:                            35.47     
==================================================
```


FP8 quantization benchmark results (`fp8_c32`, dynamic FP8 verifier):

```text
============ Serving Benchmark Result ============
Successful requests:                     640       
Failed requests:                         0         
Maximum request concurrency:             32        
Benchmark duration (s):                  28.72     
Total input tokens:                      50236     
Total generated tokens:                  163840    
Request throughput (req/s):              22.28     
Output token throughput (tok/s):         5704.46   
Peak output token throughput (tok/s):    6061.00   
Peak concurrent requests:                64.00     
Total token throughput (tok/s):          7453.55   
---------------Time to First Token----------------
Mean TTFT (ms):                          52.86     
Median TTFT (ms):                        51.42     
P99 TTFT (ms):                           93.31     
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          5.41      
Median TPOT (ms):                        5.41      
P99 TPOT (ms):                           5.56      
---------------Inter-token Latency----------------
Mean ITL (ms):                           5.41      
Median ITL (ms):                         5.33      
P99 ITL (ms):                            7.81      
==================================================
```


FP8 + speculative decoding benchmark results (`fp8_spec_n1_c32`, dynamic FP8 verifier plus EAGLE-3 draft head, `num_speculative_tokens=1`):

```text
============ Serving Benchmark Result ============
Successful requests:                     640       
Failed requests:                         0         
Maximum request concurrency:             32        
Benchmark duration (s):                  24.94     
Total input tokens:                      50236     
Total generated tokens:                  163840    
Request throughput (req/s):              25.66     
Output token throughput (tok/s):         6569.96   
Peak output token throughput (tok/s):    5127.00   
Peak concurrent requests:                64.00     
Total token throughput (tok/s):          8584.41   
---------------Time to First Token----------------
Mean TTFT (ms):                          31.18     
Median TTFT (ms):                        26.89     
P99 TTFT (ms):                           93.33     
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          4.68      
Median TPOT (ms):                        4.69      
P99 TPOT (ms):                           5.21      
---------------Inter-token Latency----------------
Mean ITL (ms):                           6.33      
Median ITL (ms):                         6.06      
P99 ITL (ms):                            16.20     
---------------Speculative Decoding---------------
Acceptance rate (%):                     35.41     
Acceptance length:                       1.35      
Drafts:                                  120655    
Draft tokens:                            120655    
Accepted tokens:                         42718     
Per-position acceptance (%):
  Position 0:                            35.41     
==================================================
```


## Appendix: Final Sweep Notes

The final benchmark sweep produced `40` measured benchmark rows plus `10` warmup rows. The selected final runs were the highest-output-throughput runs for their groups:

| Group | Selected run | Output tok/s | Why selected |
| --- | --- | ---: | --- |
| Baseline BF16 | `baseline_bf16_c32` | `4215.67` | Highest baseline throughput in the concurrency sweep |
| EAGLE-3 BF16 | `spec_bf16_n1_c32` | `5040.63` | Highest BF16 speculative throughput and best acceptance/TPOT tradeoff |
| FP8 | `fp8_c32` | `5704.46` | Highest FP8 throughput |
| FP8 + EAGLE-3 | `fp8_spec_n1_c32` | `6569.96` | Highest combined throughput and best acceptance/TPOT tradeoff |

The first low-load benchmark attempt did not pass thresholds because it underutilized the H100:

| Configuration | Initial output tok/s | Final output tok/s | Improvement |
| --- | ---: | ---: | ---: |
| Baseline BF16 | `838.44` | `4215.67` | `5.03x` |
| EAGLE-3 BF16 | `1201.13` | `5040.63` | `4.20x` |
| FP8 | `1069.66` | `5704.46` | `5.33x` |
| FP8 + EAGLE-3 | `1628.42` | `6569.96` | `4.03x` |

The model artifacts did not need to be replaced. The final improvement came mostly from benchmarking with enough load for vLLM batching: concurrency `32`, `640` measured requests, `max_num_seqs=64`, and `max_num_batched_tokens=32768`.
